In [ ]:
# SCENARIO: “AI Banking Assistant with Role-Based Access”
# 🏦 Background Story

# A bank builds an AI assistant to help users:

# Check account balance
# View transactions
# Approve loans
# Manage customer accounts

# 👉 But not everyone can do everything

# 🧑‍🤝‍🧑 Roles in the Bank
# Role	Description
# 👤 Customer	Bank account holder
# 👨‍💼 Employee	Bank staff
# 🧑‍💼 Manager	Branch manager
# 🔐 Permissions (RBAC)
# Action	Customer	Employee	Manager
# View own balance	✅	✅	✅
# View others' accounts	❌	✅	✅
# Approve loan	❌	❌	✅
# View all transactions	❌	✅	✅


# ======================================
# STEP 1: Install Libraries
# ======================================

!pip install groq gradio


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("❌ GROQ_API_KEY not found in Colab Secrets")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Bank Database
# ======================================
accounts = {
    "1001": {"name": "Amit", "balance": 50000},
    "1002": {"name": "Neha", "balance": 75000}
}


# ======================================
# STEP 4: Tool Functions (APIs)
# ======================================
def get_balance(account_id):
    if account_id in accounts:
        return f"💰 Balance of {account_id}: ₹{accounts[account_id]['balance']}"
    return "❌ Account not found"


def approve_loan(account_id):
    if account_id in accounts:
        return f"🏦 Loan approved for account {account_id}"
    return "❌ Account not found"


# ======================================
# STEP 5: RBAC Security Layer
# ======================================
def secure_access(role, user_account, requested_account, action):

    # Manager → full access
    if role == "manager":
        return True

    # Employee → can view all but cannot approve loans
    elif role == "employee":
        if action == "approve_loan":
            return False
        return True

    # Customer → only own account, no loan approval
    elif role == "customer":
        return user_account == requested_account and action != "approve_loan"

    return False


# ======================================
# STEP 6: MCP Tool Decision via LLM
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        You are an AI banking assistant.

        Decide which action to take:
        - get_balance
        - approve_loan

        Rules:
        - Balance related queries → get_balance
        - Loan approval queries → approve_loan

        Return ONLY the action name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        action = response.choices[0].message.content.strip().lower()
        return action

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 7: MCP Agent (Core Logic)
# ======================================
def banking_agent(message, role, user_account, requested_account, history):

    # Input validation
    if not user_account:
        response = "⚠️ Please enter your account ID"
        history.append((message, response))
        return "", history

    if not requested_account:
        requested_account = user_account  # default to own account

    # Step 1: LLM decides action
    action = decide_action(message)

    # Step 2: RBAC Security Check
    if not secure_access(role, user_account, requested_account, action):
        response = "🚫 Access Denied: You are not authorized"

    else:
        # Step 3: Tool Invocation
        if "balance" in action:
            response = get_balance(requested_account)

        elif "loan" in action:
            response = approve_loan(requested_account)

        # Fallback if LLM fails
        elif action == "fallback":
            msg_lower = message.lower()
            if "balance" in msg_lower:
                response = get_balance(requested_account)
            elif "loan" in msg_lower:
                response = approve_loan(requested_account)
            else:
                response = "⚠️ Could not understand request"

        else:
            response = "🤖 Try asking about balance or loan"

    # Save chat history
    history.append((message, response))

    return "", history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 AI Banking Assistant (MCP + RBAC + Groq)")

    role = gr.Dropdown(
        ["customer", "employee", "manager"],
        label="Select Role"
    )

    user_account = gr.Textbox(label="Your Account ID (e.g., 1001)")
    requested_account = gr.Textbox(label="Target Account ID (optional)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, role, user_account, requested_account, state],
        outputs=[msg, chatbot]
    )


# ======================================
# STEP 9: Launch App
# ======================================
demo.launch(share=True)

/tmp/ipykernel_579/3234274250.py:191: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_579/3234274250.py:191: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6a353bd95e32fc3ca6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
# ======================================
# STEP 1: Load ENV
# ======================================
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)


# ======================================
# STEP 2: MOCK MCP TOOLS
# ======================================
import asyncio
import random
import nest_asyncio

nest_asyncio.apply()


async def web_search(query):

    await asyncio.sleep(1)

    return f"📰 News about {query}: Market is growing fast."


async def get_stock_data(company):

    await asyncio.sleep(1)

    price = random.randint(100, 500)

    return f"📈 Stock price of {company}: ${price}"


async def fetch_company_profile(company):

    await asyncio.sleep(1)

    return f"👥 {company} has 5000 employees, HQ in USA"


# ======================================
# STEP 3: PARALLEL TOOL CALLS
# ======================================
async def parallel_research(company):

    results = await asyncio.gather(

        web_search(company),

        get_stock_data(company),

        fetch_company_profile(company),

        return_exceptions=True
    )

    news, stock, profile = results


    return {

        "news": news,

        "stock": stock,

        "profile": profile
    }


# ======================================
# STEP 4: LLM ANALYSIS (NVIDIA)
# ======================================
def analyse_text(text):

    response = client.chat.completions.create(

        model="meta/llama3-70b-instruct",

        messages=[

            {

                "role": "user",

                "content": f"""
                Analyze this business data
                and give key insights:

                {text}
                """

            }

        ]

    )

    return response.choices[0].message.content



# ======================================
# STEP 5: REPORT GENERATION (NVIDIA)
# ======================================
def generate_report(analysis, company):

    response = client.chat.completions.create(

        model="meta/llama3-70b-instruct",

        messages=[

            {

                "role": "user",

                "content": f"""
                Create professional report
                for {company}:

                {analysis}
                """

            }

        ]

    )

    return response.choices[0].message.content


# ======================================
# STEP 6: FULL MCP PIPELINE
# ======================================
async def full_pipeline(company):

    # parallel tools
    data = await parallel_research(company)


    combined_text = f"""

    News:
    {data['news']}

    Stock:
    {data['stock']}

    Profile:
    {data['profile']}
    """


    # chain LLM calls
    analysis = analyse_text(combined_text)

    report = generate_report(analysis, company)


    return report


# ======================================
# STEP 7: RUN
# ======================================
company_name = "Tesla"

result = asyncio.run(

    full_pipeline(company_name)

)

print("\n📊 FINAL REPORT:\n")

print(result)


📊 FINAL REPORT:

**Tesla Business Report**

**Executive Summary**

This report provides a comprehensive analysis of Tesla's business performance, highlighting key insights and trends that are shaping the company's future prospects. Our analysis reveals a positive market trend, strong stock performance, and a significant scale and presence in the USA. These factors suggest that Tesla is well-positioned to capitalize on the growing demand for electric vehicles and drive revenue growth.

**Market Analysis**

The electric vehicle market is experiencing rapid growth, driven by increasing demand, government incentives, and environmental concerns. This trend is expected to continue, presenting opportunities for Tesla to expand its operations and increase revenue.

**Stock Performance**

Tesla's stock price stands at $481, indicating strong investor confidence in the company's growth potential. This performance is likely driven by the positive market trend, strong financials, and the company'